In [ ]:
import json
import os
from pathlib import Path

from ml441_assignment4.experiment import pilot_run
from ml441_assignment4.primary import run_primary
from ml441_assignment4.records import canonical_hash, file_hash
from ml441_assignment4.search import structural_candidates
from ml441_assignment4.verify_results import verify_manifest
from publication_inputs import inspect_inputs

AUTOMATIC_COMMITS = False
AUTOMATIC_PUSHES = False
RUN_PILOT = False
RUN_PRIMARY = False
PROJECT_ROOT = Path(os.environ.get("ML441_A4_PROJECT_ROOT", Path.cwd())).expanduser().resolve()


In [ ]:
inventory = inspect_inputs(PROJECT_ROOT, phase="primary")
print(json.dumps(inventory, indent=2))


In [ ]:
protocol_path = PROJECT_ROOT / "configs/primary_protocol.json"
protocol = json.loads(protocol_path.read_text()) if protocol_path.is_file() else None
if protocol is not None:
    assert protocol["status"] == "frozen"
    assert inventory["protocol_seal_matches"] is True
    print(json.dumps({"status": protocol["status"], "seeds": protocol["ensemble"]["seeds"], "selection": protocol["selection"], "structural_candidates": structural_candidates(protocol)}, indent=2))
else:
    print("Provide the original local evidence project through ML441_A4_PROJECT_ROOT to inspect its frozen protocol.")


In [ ]:
if RUN_PILOT:
    pilot_summary = pilot_run(PROJECT_ROOT, workers=4)
else:
    print("Development pilot execution is disabled; its separate provisional protocol is a required local input when explicitly enabled.")


In [ ]:
latest_path = PROJECT_ROOT / "output/pilot/latest.json"
if latest_path.is_file():
    latest = json.loads(latest_path.read_text())
    directory = PROJECT_ROOT / latest["directory"]
    assert file_hash(directory / "summary.json") == latest["summary_sha256"]
    print(verify_manifest(directory))
else:
    print("No saved development pilot is available in the selected local project.")


In [ ]:
if RUN_PRIMARY:
    if protocol is None:
        raise ValueError("The frozen primary protocol and its local inputs are required")
    run_primary(PROJECT_ROOT, mode="all")
else:
    print("Primary execution is disabled. Enabling it retains every frozen data, split, code, environment and evidence gate.")


In [ ]:
if protocol is not None:
    summary_path = PROJECT_ROOT / "output/primary" / canonical_hash(protocol)[:16] / "comparison_summary.json"
    if summary_path.is_file():
        summary = json.loads(summary_path.read_text())
        print(json.dumps({key: summary[key] for key in ["status", "macro_f1_fold_mean", "macro_f1_fold_sd", "folds", "paired_comparisons", "inference"]}, indent=2))
    else:
        print("No complete primary comparison is available in the selected local project.")
